#### All Global Imports

In [39]:
import pandas as pd

### Load the data set

In [40]:
# url = "https://raw.githubusercontent.com/fivethirtyeight/data/master/alcohol-consumption/drinks.csv"
# df = pd.read_csv(url)
df = pd.read_csv("../assets/drinks.csv")
print(df.head())

       country  beer_servings  spirit_servings  wine_servings  \
0  Afghanistan              0                0              0   
1      Albania             89              132             54   
2      Algeria             25                0             14   
3      Andorra            245              138            312   
4       Angola           -217               57             45   

   total_litres_of_pure_alcohol continent  
0                           0.0      Asia  
1                           4.9    Europe  
2                           0.7    Africa  
3                          12.4    Europe  
4                           5.9    Africa  


#### Initial Inspection (Get the shape)

In [41]:
print("Total rows (Countries) ", df.shape[0])
print("Total columns (Features) ", df.shape[1])

Total rows (Countries)  193
Total columns (Features)  6


### Schema Health Checking
#### The Goal: Ensure every column has the correct data type.
- Check df.info() to see if numbers are stored correctly.

- Idea: Look at columns like beer_servings, spirit_servings, wine_servings, and total_litres_of_pure_alcohol. Are they stored as floats/integers, or did any accidental text strings slip in?

In [42]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 193 entries, 0 to 192
Data columns (total 6 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   country                       193 non-null    object 
 1   beer_servings                 193 non-null    int64  
 2   spirit_servings               193 non-null    int64  
 3   wine_servings                 193 non-null    int64  
 4   total_litres_of_pure_alcohol  193 non-null    float64
 5   continent                     192 non-null    object 
dtypes: float64(1), int64(3), object(2)
memory usage: 9.2+ KB
None


#### Spotting Missing Values & Imputation
- The Goal: Handle missing data gracefully.
- Idea: If a continent is missing for a country, can you write a simple rule or imputation step to fill it based on the country name?

In [43]:
missing_values = df.isnull().sum()
print(missing_values)

country                         0
beer_servings                   0
spirit_servings                 0
wine_servings                   0
total_litres_of_pure_alcohol    0
continent                       1
dtype: int64


####  DATA SANITIZATION

In [44]:
df['continent'] = df['continent'].fillna('Unknown')


In [45]:

# print(df[df['country'] == 'Sweden']) # boolean indexing
# print(df.query("country == 'Sweden'")[['country','continent']])
# print(df.loc[df['country'] == 'Sweden', ['country','continent']])
print(df[df["country"].isin(["Sweden","India"])][['country','continent']])


    country continent
77    India      Asia
165  Sweden   Unknown


#### Handling Outliers and Anomalies
- The Goal: Clean out unrealistic numbers using logical sanity checks.
- Check df.describe() to look at minimum and maximum values.
- Idea: Filter out or investigate rows where alcohol servings are negative, or where total_litres_of_pure_alcohol doesn't mathematically align with the servings.

In [46]:
print(df.describe().T[['mean','std','min','max']])

                                    mean         std    min    max
beer_servings                 103.911917  103.463900 -217.0  376.0
spirit_servings                80.994819   88.284312    0.0  438.0
wine_servings                  49.450777   79.697598    0.0  370.0
total_litres_of_pure_alcohol    4.717098    3.773298    0.0   14.4


In [51]:
print("negative beer servings rows :\n",df[df['beer_servings'] < 0][['country','beer_servings']])
beer_serving_negative_count= (df['beer_servings'] < 0).sum()
print("Beer serving negative count:",beer_serving_negative_count)

negative beer servings rows :
   country  beer_servings
4  Angola           -217
Beer serving negative count: 1


### Lets fix negative , either feed zero or filter out countries with negative serving

In [52]:
df_copy = df.copy()
print("df dimension before removing negative beer servings:", df.shape)
df = df[df['beer_servings'] >= 0]
print("df dimension after removing negative beer servings:", df.shape)

df dimension before removing negative beer servings: (193, 6)
df dimension after removing negative beer servings: (192, 6)


### string sanitization

In [56]:
# get all features names from dataframe
feature_names = df.columns
print(feature_names)
# assume Identifying leading or trailing spaces in feature name "continent"
df['continent'] = df['continent'].str.strip().str.title()
print(df['continent'].head())

Index(['country', 'beer_servings', 'spirit_servings', 'wine_servings',
       'total_litres_of_pure_alcohol', 'continent'],
      dtype='object')
0             Asia
1           Europe
2           Africa
3           Europe
5    North America
Name: continent, dtype: object


In [ ]:
print(df['continent'].unique())

['Asia' 'Europe' 'Africa' 'North America' 'South America' 'Oceania'
 'Unknown']


### Grouping and Aggregation
- The Goal: Answer business/analytical questions with simple aggregations.
- Which continent consumes the most beer on average?
- Which country has the absolute highest total_litres_of_pure_alcohol?

In [59]:
average_beer_by_continent = df.groupby('continent')['beer_servings'].mean().round(2)
print(average_beer_by_continent)

continent
Africa            58.48
Asia              37.05
Europe           194.73
North America    145.43
Oceania           89.69
South America    175.08
Unknown          152.00
Name: beer_servings, dtype: float64


In [60]:
top_drinker = df.loc[df['total_litres_of_pure_alcohol'].idxmax()]
print(top_drinker)

country                         Belarus
beer_servings                       142
spirit_servings                     373
wine_servings                        42
total_litres_of_pure_alcohol       14.4
continent                        Europe
Name: 15, dtype: object


### Summary on this notebook
- 1. Inspecting Structure (head() & info()): How to quickly check row counts, columns, and data types to catch unexpected text fields.

- 2. Finding Dirt (isnull()): Spotting missing data and learning how to handle it safely (fillna()).

- 3. Text Cleaning (str.strip()): Normalizing string categories so grouping operations don't break due to hidden trailing spaces or typos.

- 4. Sanity Checks (< 0 filters): Using simple logical conditions to ensure data falls within realistic bounds.

- 5. Summarizing & Grouping (describe() & groupby()): Turning thousands of rows into actionable answers like regional averages and extreme outliers.